In [ ]:
! pip install -q kaggle

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

print("Reading dataset")

target = 'claim'


train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
train.pop('id')
test.pop('id')

train["Set"] = np.random.choice(["train", "valid"], p =[.9, .1], size=(train.shape[0],))

train_indices = train[train.Set=="train"].index
valid_indices = train[train.Set=="valid"].index

print("Generating new features")

train['n_missing'] = train.isna().sum(axis=1)
"""
na_one_hot = train.iloc[:, :118].isna()
na_one_hot.columns = [f'missing_f_{i}' for i in range(118)]
train = pd.concat([train, na_one_hot], axis=1)
"""

test['n_missing'] = test.isna().sum(axis=1)
"""
na_one_hot = test.iloc[:, :118].isna()
na_one_hot.columns = [f'missing_f_{i}' for i in range(118)]
test = pd.concat([test, na_one_hot], axis=1)
"""

print(train.columns)

print("Preprocessing data")

from sklearn.preprocessing import QuantileTransformer,  KBinsDiscretizer, RobustScaler , StandardScaler
from sklearn.decomposition import PCA
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import SimpleImputer, IterativeImputer
from sklearn.pipeline import Pipeline

"""
pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='mean',missing_values=np.nan)),
        ("scaler", QuantileTransformer(n_quantiles=2048, output_distribution='uniform')),
        ('bin', KBinsDiscretizer(n_bins=2048, encode='ordinal',strategy='uniform'))
        ])
"""

pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='median',missing_values=np.nan)),
        #('imputer', IterativeImputer(missing_values=np.nan)),
        ("scaler", StandardScaler()),
        #("pca", PCA(n_components=20)),
        ('bin', KBinsDiscretizer(n_bins=4096, encode='ordinal',strategy='uniform'))
        ])

#generated_features = [f'missing_f_{i}' for i in range(118)]
#column_X = [col for col in train.columns if col not in generated_features+(['claim', 'Set'])]
column_X = [col for col in train.columns if col not in ['claim', 'Set', 'n_missing']]
print(column_X)

train_transform = pd.DataFrame()
test_transform = pd.DataFrame()

train_transform['claim'] = train['claim'].to_numpy()
train_transform['Set'] = train['Set'].to_numpy()
train_transform['n_missing'] = train['n_missing'].to_numpy()
test_transform['n_missing'] = test['n_missing'].to_numpy()

train_transform = pd.concat([train_transform, pd.DataFrame(pipe.fit_transform(train[column_X]))], axis=1)
test_transform = pd.concat([test_transform, pd.DataFrame(pipe.transform(test[column_X]))], axis=1)

#print(train_transform.columns)
#print(test_transform.columns)

#train_transform['claim'] = train['claim'].to_numpy()
#train_transform['Set'] = train['Set'].to_numpy()
#train_transform['n_missing'] = train['n_missing'].to_numpy()
#test_transform['n_missing'] = test['n_missing'].to_numpy()

#train_transform.to_csv('preprocessed_train2.csv')
#test_transform.to_csv('preprocessed_test2.csv')


"""
nunique = train.nunique()
types = train.dtypes

categorical_columns = []
categorical_dims =  {}
for col in train.columns:
    if types[col] == 'object' or nunique[col] < 300:
        l_enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
        train[col] = train[col].fillna("VV_likely")
        train[col] = l_enc.fit_transform(train[col].values.reshape(-1, 1))
        if col in test.columns: test[col] = l_enc.transform(test[col].values.reshape(-1, 1))
        categorical_columns.append(col)
        categorical_dims[col] = len(l_enc.categories_[0])
    else:
        train.fillna(train.loc[train_indices, col].mean(), inplace=True)
"""

#train = pd.read_csv('preprocessed_train.csv', index_col=0)
#test = pd.read_csv('preprocessed_test.csv', index_col=0)
#train_indices = train[train.Set=="train"].index
#valid_indices = train[train.Set=="valid"].index

unused_feat = ['Set']
features = [ col for col in train_transform.columns if col not in unused_feat+[target]]
cat_idxs = [i for i in range(len(features))]
cat_dims = [4096 for i in range(len(features))]
#cat_dims = [256 for i in range(118)] + [119] + [2 for i in range(118)]
#cat_idxs = [ i for i, f in enumerate(features) if f in categorical_columns]
#cat_dims = [ categorical_dims[f] for i, f in enumerate(features) if f in categorical_columns]

#pri7nt(cat_idxs)
#print(cat_dims)

x_train = train_transform[features].values[train_indices]
y_train = train_transform[target].values[train_indices]

x_val = train_transform[features].values[valid_indices]
y_val = train_transform[target].values[valid_indices]

x_train = np.asarray(x_train, dtype=np.int64)
y_train = np.asarray(y_train, dtype=np.int64)
x_val = np.asarray(x_val, dtype=np.int64)
y_val = np.asarray(y_val, dtype=np.int64)

x_test = test_transform[features].to_numpy(dtype=np.int64)

print(x_train)
print(x_test)

print("X train shape: ", x_train.shape)
print("X validation shape: ", x_val.shape)
print("X test shape: ", x_test.shape)
print("Y train shape: ", y_train.shape)
print("Y validation shape: ", y_val.shape)

print("number of zero label in train data: ", np.sum(y_train)/float(x_train.shape[0]))
print("number of zero label in validation data: ", np.sum(y_val)/float(x_val.shape[0]))

###################

import pytorch_tabnet
from pytorch_tabnet.tab_model import TabNetClassifier
from pytorch_tabnet.pretraining import TabNetPretrainer
import torch

from sklearn.metrics import roc_auc_score, accuracy_score,log_loss

"""
print("Pretraining unsupervised model")

unsupervised_model = TabNetPretrainer(cat_idxs=cat_idxs,
                       cat_dims=cat_dims,
                       cat_emb_dim=2,
                       n_d = 32,
                       n_a = 32,
                       gamma = 1.3,
                       n_steps = 3,
                       lambda_sparse = 1e-3,
                       optimizer_fn=torch.optim.Adam,
                       optimizer_params=dict(lr=2e-2),
                       mask_type='entmax' # "sparsemax"
                      )

unsupervised_model.fit(
    X_train=x_train,
    eval_set=[x_val],
    pretraining_ratio=0.8,
    batch_size=1024, virtual_batch_size=128,
    max_epochs = 100,
)

saving_path_name = "pretrained_unsupervised6-ensemble-x2"
saved_filepath = unsupervised_model.save_model(saving_path_name)
"""


print("Training supervised model")

"""
model = TabNetClassifier(cat_idxs=cat_idxs,
                       cat_dims=cat_dims,
                       cat_emb_dim=2,
                       n_d = 12,
                       n_a = 12,
                       n_steps = 4,
                       lambda_sparse = 1e-3,
                       optimizer_fn=torch.optim.Adam,
                       optimizer_params=dict(lr=2e-2, weight_decay = 1e-4),
                       scheduler_params={"step_size":1, # how to use learning rate scheduler
                                         "gamma":0.9},
                       scheduler_fn=torch.optim.lr_scheduler.StepLR,
                       mask_type='entmax' # "sparsemax"
                      )
"""

model = TabNetClassifier(cat_idxs=cat_idxs,
                       cat_dims=cat_dims,
                       cat_emb_dim=2,
                       n_d = 16,
                       n_a = 16,
                       gamma = 1.3,
                       n_steps = 3,
                       lambda_sparse = 1e-3,
                       optimizer_fn=torch.optim.AdamW,
                       optimizer_params=dict(lr=1e-3, weight_decay = 1e-2),
                       scheduler_params={"step_size":5, # how to use learning rate scheduler
                                         "gamma":0.9},
                       scheduler_fn=torch.optim.lr_scheduler.StepLR,
                       mask_type='entmax' # "sparsemax"
                      )

model.fit(
    x_train,y_train,
    eval_set=[(x_train, y_train), (x_val, y_val)],
    eval_name=['train', 'valid'],
    eval_metric=['auc'],
    max_epochs=100 , patience=200,
    #from_unsupervised=unsupervised_model,
    loss_fn=torch.nn.functional.cross_entropy,
    batch_size=8096, virtual_batch_size=2048,
    weights=1,
    drop_last=False
)

saving_path_name2 = "supervised-roc-2"
saved_filepath = model.save_model(saving_path_name2)

print(model.feature_importances_)

saving_path_name2 = "supervised-roc-2"
load_model = TabNetClassifier()
load_model.load_model(saving_path_name2 + ".zip")

"""
pred_tabnet_train = load_model.predict_proba(x_train)[:, 0].reshape(-1,1)
pred_tabnet_test = load_model.predict_proba(x_test)[:, 0].reshape(-1,1)
print(pred_tabnet_test)
"""

#########################
"""
from xgboost import XGBClassifier


xgb_params = {
      'tree_method' : 'gpu_hist',
      'learning_rate' : 0.01,
      'n_estimators' : 50000,
      'colsample_bytree' : 0.3,
      'subsample' : 0.75,
      'reg_alpha' : 19,
      'reg_lambda' : 19,
      'max_depth' : 5,
      'predictor' : 'gpu_predictor'
}

model_xgb = XGBClassifier(**xgb_params)

model_xgb.fit(x_train, y_train,
        eval_set=[(x_val, y_val)],
        early_stopping_rounds=200,
        verbose=1000)

model_xgb.save_model("model_xgb-ensemble-3.json")

model_xgb = XGBClassifier()
model_xgb.load_model("model_xgb-ensemble-3.json")

pred_xgb_train = model_xgb.predict_proba(x_train, ntree_limit=model_xgb.best_ntree_limit)[:, 0].reshape(-1,1)
pred_xgb_test = model_xgb.predict_proba(x_test, ntree_limit=model_xgb.best_ntree_limit)[:, 0].reshape(-1,1)

print(pred_xgb_test)
"""

"""
#########################
from lightgbm import LGBMClassifier

lgb1_params = {
    'objective': 'binary',
    'n_estimators': 10000,
    'random_state': 42,
    'learning_rate': 0.095,
    'subsample': 0.6,
    'subsample_freq': 1,
    'colsample_bytree': 0.4,
    'reg_alpha': 10.0,4
    'reg_lambda': 1e-1,
    'min_child_weight': 256,
    'min_child_samples': 20,
    'device' : 'gpu',
    'max_depth' : 3,
    'num_leaves' : 7
}

model_lgb1 = LGBMClassifier(**lgb1_params)

model_lgb1.fit(x_train, y_train,
        eval_set=[(x_val, y_val)],
        early_stopping_rounds=200,
        verbose=1000)

model_lgb1.save_model("model_lgb1-ensemble-3.json")

model_lgb1 = LGBMClassifier()
model_lgb1.load_model("model_lgb1-ensemble-3.json")

pred_lgb1_train = model_lgb1.predict_proba(x_train)[:, 0].reshape(-1,1)
pred_lgb1_test = model_lgb1.predict_proba(x_test)[:, 0].reshape(-1,1)

print(pred_lgb1_test)

#########################

lgb2_params = {
    'max_depth' : 3,
    'num_leaves' : 7,
    'n_estimators' : 5000,
    'colsample_bytree' : 0.3,
    'subsample' : 0.5,
    'random_state' : 42,
    'reg_alpha' : 18,
    'reg_lambda' : 17,
    'learning_rate' : 0.095,
    'device' : 'gpu',
    'objective' : 'binary'
}

model_lgb2 = LGBMClassifier(**lgb2_params)

model_lgb2.fit(x_train, y_train,
        eval_set=[(x_val, y_val)],
        early_stopping_rounds=200,
        verbose=1000)

model_lgb2.save_model("model_lgb2-ensemble-3.json")

model_lgb2 = LGBMClassifier()
model_lgb2.load_model("model_lgb2-ensemble-3.json")

pred_lgb2_train = model_lgb2.predict_proba(x_train)[:, 0].reshape(-1,1)
pred_lgb2_test = model_lgb2.predict_proba(x_test)[:, 0].reshape(-1,1)

print(pred_lgb2_test)
"""

#######################3

"""
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

sc = StandardScaler()

x_train = np.concatenate([sc.fit_transform(x_train), pred_tabnet_train, pred_xgb_train], axis=1)
x_test = np.concatenate([sc.transform(x_test), pred_tabnet_test, pred_xgb_test], axis=1)

lr = LogisticRegression(random_state = 0, C = 1000., max_iter = 1000)
lr.fit(x_train, y_train)

test_pred = lr.predict_proba(x_test)
"""

test_pred = load_model.predict_proba(x_test)

print(test_pred)

submission = pd.read_csv('sample_solution.csv')
submission['claim'] = test_pred[:, 1]
submission.to_csv('supervised-roc-2.csv', index = 0)


In [ ]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"sincostanx","key":"53de394d239fc0ec9825d27f2c4f8d04"}'}

In [ ]:
! mkdir ~/.kaggle
! cp kaggle.json ~/.kaggle/
! chmod 600 ~/.kaggle/kaggle.json
! kaggle datasets list

ref                                                         title                                              size  lastUpdated          downloadCount  
----------------------------------------------------------  ------------------------------------------------  -----  -------------------  -------------  
gpreda/reddit-vaccine-myths                                 Reddit Vaccine Myths                              233KB  2021-09-03 10:07:19          11697  
crowww/a-large-scale-fish-dataset                           A Large Scale Fish Dataset                          3GB  2021-04-28 17:03:01           7082  
imsparsh/musicnet-dataset                                   MusicNet Dataset                                   22GB  2021-02-18 14:12:19           2751  
dhruvildave/wikibooks-dataset                               Wikibooks Dataset                                   2GB  2021-07-03 18:37:20           2813  
nickuzmenkov/nih-chest-xrays-tfrecords                      NIH Chest X-rays

In [ ]:
! kaggle competitions download -c tabular-playground-series-sep-2021

  0% 0.00/1.10M [00:00<?, ?B/s]
100% 1.10M/1.10M [00:00<00:00, 73.2MB/s]
 98% 385M/394M [00:05<00:00, 62.8MB/s]
100% 394M/394M [00:06<00:00, 67.4MB/s]
 97% 196M/203M [00:03<00:00, 51.0MB/s]
100% 203M/203M [00:03<00:00, 62.6MB/s]


In [ ]:
!unzip train.csv.zip
!unzip test.csv.zip

Archive:  train.csv.zip
  inflating: train.csv               
Archive:  test.csv.zip
  inflating: test.csv                


# Section

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

In [ ]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

In [ ]:
dataset = pd.concat([train, test]).drop(columns = ['id', 'claim'])

In [ ]:
dataset['n_missing'] = dataset.isna().sum(axis=1)
dataset['std'] = dataset.std(axis=1)
dataset['min'] = dataset.min(axis=1)
dataset['max'] = dataset.max(axis=1)

# imputer = KNNImputer(missing_values=np.nan)
# imputer.fit(dataset)
# dataset_trans = imputer.transform(dataset)

In [ ]:
na_one_hot = dataset.iloc[:, :118].isna()
na_one_hot.columns = [f'missing_f_{i}' for i in range(118)]
dataset = pd.concat([dataset, na_one_hot], axis=1)

In [ ]:
x_train = dataset[:len(train)]
x_test = dataset[len(train):]
y_train = train['claim']

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

sc = StandardScaler()
si = SimpleImputer()

x_train.iloc[:, :118] = si.fit_transform(sc.fit_transform(x_train.iloc[:, :118]))
x_test.iloc[:, :118] = si.fit_transform(sc.fit_transform(x_test.iloc[:, :118]))

# rb = RobustScaler()
# dataset_trans = rb.fit_transform(dataset)
# print('Missing: %d' % sum(isnan(dataset_trans).flatten()))

/usr/local/lib/python3.7/dist-packages/pandas/core/indexing.py:1734: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  isetter(loc, value[:, i].tolist())
/usr/local/lib/python3.7/dist-packages/pandas/core/indexing.py:1734: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  isetter(loc, value[:, i].tolist())


In [ ]:
x_train = x_train.fillna(0)
x_test = x_test.fillna(0)

In [ ]:
# dataset_trans.to_csv("train_preprocessed.csv", encoding='utf-8', index=False)
# dataset_trans = np.nan_to_num(dataset_trans)
#print(dataset_trans)

In [ ]:
# print(train_label.isnull().sum())
x_train, x_val, y_train, y_val = train_test_split(x_train, y_train, test_size=0.10, random_state=42)

In [ ]:
print("X train shape: ", x_train.shape)
print("X validation shape: ", x_val.shape)
print("X test shape: ", x_test.shape)
print("Y train shape: ", y_train.shape)
print("Y validation shape: ", y_val.shape)

X train shape:  (862127, 240)
X validation shape:  (95792, 240)
X test shape:  (493474, 240)
Y train shape:  (862127,)
Y validation shape:  (95792,)


In [ ]:
import gc
del train, test, dataset, na_one_hot
gc.collect()

88

# Training Ensemble

In [ ]:
! pip install mlens
! pip install xgboost
! pip install lightgbm
! pip install catboost

In [ ]:
def Stacking_Data_Loader(model, model_name, x_train, y_train, x_test, fold):

    stk = StratifiedKFold(n_splits = fold, random_state = 42, shuffle = True)

    train_fold_pred = np.zeros((x_train.shape[0], 1))
    test_pred = np.zeros((x_test.shape[0], fold))

    for counter, (train_index, valid_index) in enumerate(stk.split(x_train, y_train)):
        x_train, y_train = train2.iloc[train_index], y[train_index]
        x_valid, y_valid = train2.iloc[valid_index], y[valid_index]

        print('------------ Fold', counter+1, 'Start! ------------')
        if model_name == 'cat':
            model.fit(x_train, y_train, eval_set=[(x_valid, y_valid)])
        elif model_name == 'xgb':
            model.fit(x_train, y_train, eval_set=[(x_valid, y_valid)], eval_metric = 'auc', verbose = 500, early_stopping_rounds = 200)
        else:
            model.fit(x_train, y_train, eval_set=[(x_valid, y_valid)], eval_metric = 'auc', verbose = 100, early_stopping_rounds = 200)

        print('------------ Fold', counter+1, 'Done! ------------')

        train_fold_pred[valid_index, :] = model.predict_proba(x_valid)[:, 1].reshape(-1, 1)
        test_pred[:, counter] = model.predict_proba(x_test)[:, 1]

    test_pred_mean = np.mean(test_pred, axis = 1).reshape(-1, 1)

    print('Done!')

    return train_fold_pred, test_pred_mean

In [ ]:
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import BaggingClassifier

lgbm_1 = LGBMClassifier(**lgb1_params)
lgbm_2 = LGBMClassifier(**lgb2_params)

xgb = XGBClassifier(**xgb1_params)

cat = CatBoostClassifier(**cat_params)

# ababoost = AdaBoostClassifier(n_estimators=1000)
# bagging = BaggingClassifier(n_estimators=1000)

model = [lgbm_1, lgbm_2, xgb, cat]

In [ ]:
from mlens.ensemble import SuperLearner

ensemble = SuperLearner()
ensemble.add(model)
ensemble.add_meta(LogisticRegression(solver='lbfgs'))
ensemble.fit(x_train, y_train)

[MLENS] backend: threading
Custom logger is already specified. Specify more than one logger at same time is not thread safe.

0:	total: 2.29s	remaining: 9h 54m 29s
0:	total: 880ms	remaining: 3h 48m 38s


KeyboardInterrupt: ignored

In [ ]:
print(ensemble.data)
test_pred = ensemble.predict(x_test)

In [ ]:
submission = pd.read_csv('sample_solution.csv')
submission['claim'] = test_pred
submission.to_csv('submission3.csv', index = 0)

# Training Tabnet

In [ ]:
% pip install pytorch-tabnet
% pip install torch

In [ ]:
import pytorch_tabnet
from pytorch_tabnet.tab_model import TabNetClassifier
from pytorch_tabnet.pretraining import TabNetPretrainer
import torch

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score, accuracy_score

In [ ]:
unsupervised_model = TabNetPretrainer(
    seed=0,
    n_d=16,
    n_a=16,
    optimizer_fn=torch.optim.Adam,
    optimizer_params=dict(lr=2e-2),
    mask_type='entmax' # "sparsemax"
)

unsupervised_model.fit(
    X_train=x_train,
    eval_set=[x_val],
    pretraining_ratio=0.8,
    batch_size = 1024,
    max_epochs = 200,
)

Device used : cuda


KeyError: ignored

In [ ]:
saving_path_name = "pretrained_unsupervised2"
saved_filepath = unsupervised_model.save_model(saving_path_name)

In [ ]:
model = TabNetClassifier(n_d=16,
    n_a=16, optimizer_fn=torch.optim.Adam,
                       optimizer_params=dict(lr=2e-2),
                       scheduler_params={"step_size":10, # how to use learning rate scheduler
                                         "gamma":0.9},
                       scheduler_fn=torch.optim.lr_scheduler.StepLR,
                       mask_type='entmax' # "sparsemax"
                      )

model.fit(
    x_train,y_train,
    eval_set=[(x_train, y_train), (x_val, y_val)],
    eval_name=['train', 'valid'],
    eval_metric=['auc','accuracy'],
    max_epochs=200 , patience=30,
    from_unsupervised=unsupervised_model,
    weights=1,
    drop_last=False
)

In [ ]:
saving_path_name2 = "supervised2"
saved_filepath = model.save_model(saving_path_name2)

# Training XGBoost

In [ ]:
! pip install xgboost

In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import KFold
from sklearn.metrics import roc_curve, auc

kf = KFold(n_splits=10, shuffle=True, random_state=1)

pred_tmp = []
scores = []

for fold, (idx_train, idx_valid) in enumerate(kf.split(x_train)):
    x_train_split, y_train_split = x_train.iloc[idx_train], y_train.iloc[idx_train]
    x_val_split, y_val_split = x_train.iloc[idx_valid], y_train.iloc[idx_valid]

    model_xgb = XGBClassifier(max_depth=10,
        learning_rate=0.01,
        n_estimators=1000,
        verbosity=0,
        silent=None,
        objective='binary:logistic',
        booster='gbtree',
        n_jobs=-1,
        nthread=None,
        subsample=0.75,
        reg_alpha=19,
        reg_lambda=19,
        random_state=0)

    model_xgb.fit(x_train_split, y_train_split,
        eval_set=[(x_val_split, y_val_split)],
        early_stopping_rounds=100,
        verbose=10)

    # validation prediction
    pred_valid = model.predict_proba(X_valid)[:,1]
    fpr, tpr, _ = roc_curve(y_valid, pred_valid)
    score = auc(fpr, tpr)
    scores.append(score)

    print(f"Fold: {fold + 1} Score: {score}")
    print('::'*20)

    # test prediction
    y_hat = model.predict_proba(test_data)[:,1]
    pred_tmp.append(y_hat)

print(f"Overall Validation Score: {np.mean(scores)}")

[0]	validation_0-error:0.227806
Will train until validation_0-error hasn't improved in 100 rounds.
[10]	validation_0-error:0.225061
[20]	validation_0-error:0.225269
[30]	validation_0-error:0.225322
[40]	validation_0-error:0.225259


KeyboardInterrupt: ignored

In [ ]:
del model_xgb
gc.collect()

87

In [ ]:
test_pred = np.mean(np.column_stack(pred_tmp),axis=1)
submission = pd.read_csv('sample_solution.csv')
submission['claim'] = test_pred
submission.to_csv('submission3.csv', index = 0)

In [ ]:
model_xgb.save_model("model_xgb.json")

In [ ]:
! unzip sample_solution.csv

Archive:  sample_solution.csv.zip
  inflating: sample_solution.csv     


In [ ]:
pred_tabnet = model.predict_proba(x_test)

ValueError: ignored

In [ ]:
pred_xgb = model_xgb.predict_proba(x_test, ntree_limit=model_xgb.best_ntree_limit)

[[0.4117902  0.5882098 ]
 [0.8801565  0.11984351]
 [0.39811116 0.60188884]
 ...
 [0.24185276 0.75814724]
 [0.8582725  0.14172752]
 [0.25780898 0.742191  ]]


In [ ]:
test_pred = np.argmax((pred_tabnet + pred_xgb)/2., axis=1)
print(test_pred)

[1 0 1 ... 1 0 1]


In [ ]:
print(pred_tabnet)
print(pred_xgb)
print(test_pred)

[[0.3945201  0.6054799 ]
 [0.8820394  0.11796059]
 [0.41722435 0.58277565]
 ...
 [0.30922508 0.690775  ]
 [0.8993298  0.1006702 ]
 [0.30343473 0.6965653 ]]
[[0.4117902  0.5882098 ]
 [0.8801565  0.11984351]
 [0.39811116 0.60188884]
 ...
 [0.24185276 0.75814724]
 [0.8582725  0.14172752]
 [0.25780898 0.742191  ]]
[1 0 1 ... 1 0 1]


In [ ]:
submission = pd.read_csv('sample_solution.csv')
submission['claim'] = test_pred
submission.to_csv('submission2.csv', index = 0)